# Klasifikasi Teks: BiLSTM & LLM — 1-Stage dan 2-Stage
Notebook ini membangun dua arsitektur (BiLSTM dan LLM) untuk klasifikasi teks olahraga:
- 1-Stage: klasifikasi langsung ke 5 label.
- 2-Stage: biner (sepakbola vs non-sepakbola), lalu 4 liga.
Termasuk tuning hyperparameter, early stopping, dropout, evaluasi (accuracy, precision, recall, F1), confusion matrix, waktu training/inferensi, dan visualisasi.

In [14]:
# Setup & Imports
import os, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from data_prep import prepare_dataset
from metrics_utils import classification_report_np, confusion_matrix_np

In [15]:

LABELS_5CLASS = ["Liga Inggris", "Liga Indonesia", "Liga Spanyol", "Liga Italia", "Olahraga non-sepakbola"]
LEAGUE_TO_IDX = {name: i for i, name in enumerate(LABELS_5CLASS)}

# Siapkan dataset split jika belum ada
if not Path('dataset/train_5class.csv').exists():
    print('Mempersiapkan dataset splits...')
    prepare_dataset()
else:
    print('Dataset splits sudah tersedia.')

print('Selesai setup.')


Dataset splits sudah tersedia.
Selesai setup.


In [16]:
# BiLSTM — 1-Stage: klasifikasi 5 kelas dengan tuning embed_dim & lstm_units
from bilstm_1stage import run_experiments as run_bilstm_1stage
bilstm_1_out = 'experiments/bilstm_1stage'
tic = time.time()
run_bilstm_1stage(embed_grid=(64, 128), lstm_grid=(64, 128), epochs=10, out_dir=bilstm_1_out)
bilstm_1_train_time = time.time() - tic
print(f'BiLSTM 1-Stage training time: {bilstm_1_train_time:.2f}s')

bilstm_1_summary = pd.read_csv(os.path.join(bilstm_1_out, 'summary.csv'))
bilstm_1_summary

Epoch 1/10


C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 173ms/step - accuracy: 0.2308 - loss: 1.6085 - val_accuracy: 0.2917 - val_loss: 1.6058
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step - accuracy: 0.3846 - loss: 1.5844 - val_accuracy: 0.3333 - val_loss: 1.5966
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3365 - loss: 1.5567 - val_accuracy: 0.3333 - val_loss: 1.5835
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.3269 - loss: 1.5113 - val_accuracy: 0.3333 - val_loss: 1.5589
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - accuracy: 0.3269 - loss: 1.4219 - val_accuracy: 0.3333 - val_loss: 1.6238
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.3462 - loss: 1.3035 - val_accuracy: 0.2917 - val_loss: 1.4974
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.7019 - loss: 1.1561 - val_accuracy: 0.4167 - val_loss: 1.5472
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.7596 - loss: 0.9623 - val_accuracy: 0.4583 - val_loss: 1.5663
Epoch 1/10

C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 182ms/step - accuracy: 0.2596 - loss: 1.6088 - val_accuracy: 0.3333 - val_loss: 1.5992
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 88ms/step - accuracy: 0.3269 - loss: 1.5848 - val_accuracy: 0.3333 - val_loss: 1.5866
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.3269 - loss: 1.5532 - val_accuracy: 0.3333 - val_loss: 1.5655
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 78ms/step - accuracy: 0.3269 - loss: 1.4952 - val_accuracy: 0.3333 - val_loss: 1.6348
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - accuracy: 0.3269 - loss: 1.4489 - val_accuracy: 0.3333 - val_loss: 1.5189
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step - accuracy: 0.4423 - loss: 1.2658 - val_accuracy: 0.2917 - val_loss: 2.0295
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.4712 - loss: 1.1802 - val_accuracy: 0.4583 - val_loss: 1.4298
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.6923 - loss: 0.9404 - val_accuracy: 0.2917 - val_loss: 1.7835
Epoch 9/10

C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 174ms/step - accuracy: 0.2500 - loss: 1.6066 - val_accuracy: 0.4167 - val_loss: 1.5962
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.5577 - loss: 1.5657 - val_accuracy: 0.3333 - val_loss: 1.5833
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.5096 - loss: 1.5207 - val_accuracy: 0.2917 - val_loss: 1.5627
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.4712 - loss: 1.4422 - val_accuracy: 0.2917 - val_loss: 1.5230
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.4904 - loss: 1.2906 - val_accuracy: 0.2917 - val_loss: 1.5480
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.5769 - loss: 1.1051 - val_accuracy: 0.4583 - val_loss: 1.3510
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.7308 - loss: 0.9213 - val_accuracy: 0.4167 - val_loss: 1.3132
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.6923 - loss: 0.8480 - val_accuracy: 0.4167 - val_loss: 1.2800
Epoch 9/10

C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 6s 235ms/step - accuracy: 0.2596 - loss: 1.6052 - val_accuracy: 0.2917 - val_loss: 1.5940
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 138ms/step - accuracy: 0.4423 - loss: 1.5650 - val_accuracy: 0.3333 - val_loss: 1.5774
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 133ms/step - accuracy: 0.3269 - loss: 1.5095 - val_accuracy: 0.3333 - val_loss: 1.5417
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 130ms/step - accuracy: 0.3269 - loss: 1.3966 - val_accuracy: 0.2917 - val_loss: 1.5115
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 129ms/step - accuracy: 0.5288 - loss: 1.2530 - val_accuracy: 0.3750 - val_loss: 1.4634
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 126ms/step - accuracy: 0.7404 - loss: 1.0589 - val_accuracy: 0.4167 - val_loss: 1.5903
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 122ms/step - accuracy: 0.6538 - loss: 0.9003 - val_accuracy: 0.3750 - val_loss: 1.8686
Saved experiment summary to: experiments/bilstm_1stage\summary.csv
BiLSTM 1-Stage training time: 28.38s


,run,embed_dim,lstm_units,val_accuracy,val_f1_weighted,test_accuracy,test_f1_weighted
0,embed64_lstm64,64,64,0.291667,0.155556,0.375000,0.238710
1,embed64_lstm128,64,128,0.458333,0.394139,0.666667,0.541847
2,embed128_lstm64,128,64,0.416667,0.374542,0.708333,0.640852
3,embed128_lstm128,128,128,0.375000,0.276709,0.541667,0.409662


In [17]:
# BiLSTM — 2-Stage pipeline (Stage1 biner, Stage2 empat liga)
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

def load_stage_splits(split_dir='dataset'):
    train1 = pd.read_csv(os.path.join(split_dir, 'train_stage1.csv'))
    val1 = pd.read_csv(os.path.join(split_dir, 'val_stage1.csv'))
    test1 = pd.read_csv(os.path.join(split_dir, 'test_stage1.csv'))
    train2 = pd.read_csv(os.path.join(split_dir, 'train_stage2.csv'))
    val2 = pd.read_csv(os.path.join(split_dir, 'val_stage2.csv'))
    test2 = pd.read_csv(os.path.join(split_dir, 'test_stage2.csv'))
    return (train1, val1, test1), (train2, val2, test2)

def vectorize_texts(train_texts, val_texts, test_texts, num_words=20000, max_len=200):
    tok = Tokenizer(num_words=num_words, oov_token='<OOV>')
    tok.fit_on_texts(train_texts)
    x_train = pad_sequences(tok.texts_to_sequences(train_texts), maxlen=max_len, padding='post')
    x_val = pad_sequences(tok.texts_to_sequences(val_texts), maxlen=max_len, padding='post')
    x_test = pad_sequences(tok.texts_to_sequences(test_texts), maxlen=max_len, padding='post')
    vocab_size = min(num_words, len(tok.word_index) + 1)
    return x_train, x_val, x_test, tok, vocab_size

def build_binary_model(vocab_size, max_len, embed_dim=128, lstm_units=128, dropout=0.3):
    m = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True),
        Bidirectional(LSTM(lstm_units, return_sequences=False)),
        Dropout(dropout),
        Dense(1, activation='sigmoid'),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='binary_crossentropy', metrics=['accuracy'])
    return m

def build_stage2_model(vocab_size, max_len, embed_dim=128, lstm_units=128, dropout=0.3):
    m = Sequential([
        Embedding(input_dim=vocab_size, output_dim=embed_dim, mask_zero=True),
        Bidirectional(LSTM(lstm_units, return_sequences=False)),
        Dropout(dropout),
        Dense(4, activation='softmax'),
    ])
    m.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return m

def optimal_threshold(y_true_bin, y_pred_prob):
    # cari threshold terbaik untuk F1
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.1, 0.9, 17):
        y_hat = (y_pred_prob >= t).astype(int)
        tp = int(np.sum((y_true_bin == 1) & (y_hat == 1)))
        fp = int(np.sum((y_true_bin == 0) & (y_hat == 1)))
        fn = int(np.sum((y_true_bin == 1) & (y_hat == 0)))
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

# Load splits
(train1, val1, test1), (train2, val2, test2) = load_stage_splits()
# Stage1
s1_train_texts = train1['text'].astype(str).tolist(); y1_train = train1['label_stage1'].values
s1_val_texts = val1['text'].astype(str).tolist(); y1_val = val1['label_stage1'].values
s1_test_texts = test1['text'].astype(str).tolist(); y1_test = test1['label_stage1'].values
x1_train, x1_val, x1_test, tok1, vocab1 = vectorize_texts(s1_train_texts, s1_val_texts, s1_test_texts)
stage1 = build_binary_model(vocab1, max_len=200, embed_dim=128, lstm_units=128)
cb = [EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)]
t0 = time.time()
stage1.fit(x1_train, y1_train, validation_data=(x1_val, y1_val), epochs=10, batch_size=32, callbacks=cb, verbose=1)
bilstm_2_stage1_time = time.time() - t0
p_val_prob = stage1.predict(x1_val, verbose=0).ravel()
thresh, _ = optimal_threshold(y1_val, p_val_prob)
print('Threshold optimal Stage1:', thresh)

# Stage2
s2_train_texts = train2['text'].astype(str).tolist(); y2_train = train2['label_stage2'].map({
    'Liga Inggris':0, 'Liga Indonesia':1, 'Liga Spanyol':2, 'Liga Italia':3
}).values
s2_val_texts = val2['text'].astype(str).tolist(); y2_val = val2['label_stage2'].map({
    'Liga Inggris':0, 'Liga Indonesia':1, 'Liga Spanyol':2, 'Liga Italia':3
}).values
s2_test_texts = test2['text'].astype(str).tolist(); y2_test = test2['label_stage2'].map({
    'Liga Inggris':0, 'Liga Indonesia':1, 'Liga Spanyol':2, 'Liga Italia':3
}).values
x2_train, x2_val, x2_test, tok2, vocab2 = vectorize_texts(s2_train_texts, s2_val_texts, s2_test_texts)
stage2 = build_stage2_model(vocab2, max_len=200, embed_dim=128, lstm_units=128)
t1 = time.time()
stage2.fit(x2_train, y2_train, validation_data=(x2_val, y2_val), epochs=10, batch_size=32, callbacks=cb, verbose=1)
bilstm_2_stage2_time = time.time() - t1

# Pipeline prediksi ke 5 kelas
def pipeline_predict_5class(texts):
    x1 = pad_sequences(tok1.texts_to_sequences(texts), maxlen=200, padding='post')
    p1 = stage1.predict(x1, verbose=0).ravel()
    is_fb = (p1 >= thresh)
    preds = []
    if np.any(is_fb):
        x2 = pad_sequences(tok2.texts_to_sequences(np.array(texts)[is_fb].tolist()), maxlen=200, padding='post')
        p2 = np.argmax(stage2.predict(x2, verbose=0), axis=1)
    else:
        p2 = np.array([])
    idx_fb = 0
    for flag in is_fb:
        if flag:
            # Map 0..3 to league indices 0..3
            preds.append(int(p2[idx_fb]))
            idx_fb += 1
        else:
            preds.append(4)  # non-sepakbola index
    return np.array(preds)

# Evaluasi train/val/test
def eval_texts(df, label_col):
    texts = df['text'].astype(str).tolist()
    y_true = df[label_col].map(LEAGUE_TO_IDX).values
    y_pred = pipeline_predict_5class(texts)
    rep = classification_report_np(y_true, y_pred, LABELS_5CLASS)
    cm = confusion_matrix_np(y_true, y_pred, len(LABELS_5CLASS))
    return rep, cm

train5 = pd.read_csv('dataset/train_5class.csv')
val5 = pd.read_csv('dataset/val_5class.csv')
test5 = pd.read_csv('dataset/test_5class.csv')
rep_train_bilstm2, cm_train_bilstm2 = eval_texts(train5, 'label_5class')
rep_val_bilstm2, cm_val_bilstm2 = eval_texts(val5, 'label_5class')
rep_test_bilstm2, cm_test_bilstm2 = eval_texts(test5, 'label_5class')

print('BiLSTM 2-Stage — Train accuracy:', rep_train_bilstm2['accuracy'])
print('BiLSTM 2-Stage — Val accuracy:', rep_val_bilstm2['accuracy'])
print('BiLSTM 2-Stage — Test accuracy:', rep_test_bilstm2['accuracy'])

bilstm_2_times = {'stage1_train_s': bilstm_2_stage1_time, 'stage2_train_s': bilstm_2_stage2_time}
bilstm_2_times


Epoch 1/10


C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 205ms/step - accuracy: 0.7925 - loss: 0.6808 - val_accuracy: 0.8261 - val_loss: 0.6484
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 108ms/step - accuracy: 0.8491 - loss: 0.5995 - val_accuracy: 0.8261 - val_loss: 0.4986
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 109ms/step - accuracy: 0.8491 - loss: 0.4021 - val_accuracy: 0.8261 - val_loss: 0.4585
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.8491 - loss: 0.3385 - val_accuracy: 0.8261 - val_loss: 0.4251
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 104ms/step - accuracy: 0.8491 - loss: 0.2654 - val_accuracy: 0.7826 - val_loss: 0.4732
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 100ms/step - accuracy: 0.8774 - loss: 0.2280 - val_accuracy: 0.7391 - val_loss: 0.6250
Threshold optimal Stage1: 0.1
Epoch 1/10


C:\Users\fauzi\python\ujian-data\.venv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 274ms/step - accuracy: 0.2841 - loss: 1.3860 - val_accuracy: 0.3500 - val_loss: 1.3803
Epoch 2/10
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step - accuracy: 0.5568 - loss: 1.3511 - val_accuracy: 0.3500 - val_loss: 1.3701
BiLSTM 2-Stage — Train accuracy: 0.5096153846153846
BiLSTM 2-Stage — Val accuracy: 0.2916666666666667
BiLSTM 2-Stage — Test accuracy: 0.4583333333333333


{'stage1_train_s': 5.443441390991211, 'stage2_train_s': 3.5776751041412354}

In [18]:
# LLM — 1-Stage dengan scheduler (linear) dan warmup
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, Trainer, TrainingArguments
)

def load_5class_splits():
    train = pd.read_csv('dataset/train_5class.csv')
    val = pd.read_csv('dataset/val_5class.csv')
    test = pd.read_csv('dataset/test_5class.csv')
    return train, val, test

def prepare_texts_labels(df):
    texts = df['text'].astype(str).tolist()
    labels = df['label_5class'].map(LEAGUE_TO_IDX).values
    return texts, labels

def tokenize_ds(tok, texts, labels):
    enc = tok(texts, truncation=True, padding=False, max_length=256)
    enc['labels'] = labels.tolist()
    return Dataset.from_dict(enc)

def run_llm_1stage_experiments(model_name='xlm-roberta-base', lr_grid=(2e-5,5e-5), batch_grid=(8,16), num_epochs=3, warmup_ratio=0.1, scheduler_type='linear', out_dir='experiments/llm_1stage_nb'):
    os.makedirs(out_dir, exist_ok=True)
    train_df, val_df, test_df = load_5class_splits()
    train_texts, y_train = prepare_texts_labels(train_df)
    val_texts, y_val = prepare_texts_labels(val_df)
    test_texts, y_test = prepare_texts_labels(test_df)
    tok = AutoTokenizer.from_pretrained(model_name)
    train_ds = tokenize_ds(tok, train_texts, y_train)
    val_ds = tokenize_ds(tok, val_texts, y_val)
    test_ds = tokenize_ds(tok, test_texts, y_test)
    collator = DataCollatorWithPadding(tokenizer=tok)
    rows = []
    tic = time.time()
    for lr in lr_grid:
        for bs in batch_grid:
            run_name = f'lr{lr}_bs{bs}'
            run_dir = os.path.join(out_dir, run_name)
            os.makedirs(run_dir, exist_ok=True)
            model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=len(LABELS_5CLASS))
            args = TrainingArguments(
                output_dir=run_dir,
                learning_rate=lr,
                per_device_train_batch_size=bs,
                per_device_eval_batch_size=bs,
                num_train_epochs=num_epochs,
                evaluation_strategy='epoch',
                save_strategy='epoch',
                save_total_limit=2,
                load_best_model_at_end=True,
                metric_for_best_model='eval_loss',
                logging_steps=10,
                seed=42,
                report_to=[],
                fp16=torch.cuda.is_available(),
                lr_scheduler_type=scheduler_type,
                warmup_ratio=warmup_ratio,
            )
            trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds, tokenizer=tok, data_collator=collator)
            trainer.train()
            # train/val/test predictions
            train_pred = trainer.predict(train_ds); y_train_pred = np.argmax(train_pred.predictions, axis=1)
            val_pred = trainer.predict(val_ds); y_val_pred = np.argmax(val_pred.predictions, axis=1)
            test_pred = trainer.predict(test_ds); y_test_pred = np.argmax(test_pred.predictions, axis=1)
            rep_train = classification_report_np(y_train, y_train_pred, LABELS_5CLASS)
            rep_val = classification_report_np(y_val, y_val_pred, LABELS_5CLASS)
            rep_test = classification_report_np(y_test, y_test_pred, LABELS_5CLASS)
            # confusion matrices
            cm_val = confusion_matrix_np(y_val, y_val_pred, len(LABELS_5CLASS))
            cm_test = confusion_matrix_np(y_test, y_test_pred, len(LABELS_5CLASS))
            # save reports
            with open(os.path.join(run_dir, 'report_train.json'), 'w', encoding='utf-8') as f: json.dump(rep_train, f, ensure_ascii=False, indent=2)
            with open(os.path.join(run_dir, 'report_val.json'), 'w', encoding='utf-8') as f: json.dump(rep_val, f, ensure_ascii=False, indent=2)
            with open(os.path.join(run_dir, 'report_test.json'), 'w', encoding='utf-8') as f: json.dump(rep_test, f, ensure_ascii=False, indent=2)
            # plot cm
            plt.figure(figsize=(8,6)); sns.heatmap(cm_val, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS_5CLASS, yticklabels=LABELS_5CLASS); plt.tight_layout(); plt.savefig(os.path.join(run_dir, 'cm_val.png')); plt.close()
            plt.figure(figsize=(8,6)); sns.heatmap(cm_test, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS_5CLASS, yticklabels=LABELS_5CLASS); plt.tight_layout(); plt.savefig(os.path.join(run_dir, 'cm_test.png')); plt.close()
            rows.append({
                'run': run_name, 'learning_rate': lr, 'batch_size': bs,
                'train_accuracy': rep_train.get('accuracy', 0),
                'val_accuracy': rep_val.get('accuracy', 0),
                'test_accuracy': rep_test.get('accuracy', 0),
                'val_f1_weighted': rep_val.get('weighted avg', {}).get('f1-score', 0),
                'test_f1_weighted': rep_test.get('weighted avg', {}).get('f1-score', 0),
            })
    toc = time.time()
    summary = pd.DataFrame(rows)
    summary.to_csv(os.path.join(out_dir, 'summary.csv'), index=False)
    print('LLM 1-Stage total training time (grid):', f'{toc-tic:.2f}s')
    return summary

llm_1_summary = run_llm_1stage_experiments(model_name='xlm-roberta-base', lr_grid=(2e-5, 5e-5), batch_grid=(8, 16), num_epochs=3)
llm_1_summary

Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
# LLM — 2-Stage pipeline dengan threshold optimal di Stage1
from transformers import AutoModelForSequenceClassification

def run_llm_2stage(model_name='xlm-roberta-base', num_epochs=3, lr=2e-5, bs=16):
    # Load stage splits
    train1 = pd.read_csv('dataset/train_stage1.csv'); val1 = pd.read_csv('dataset/val_stage1.csv'); test1 = pd.read_csv('dataset/test_stage1.csv')
    train2 = pd.read_csv('dataset/train_stage2.csv'); val2 = pd.read_csv('dataset/val_stage2.csv'); test2 = pd.read_csv('dataset/test_stage2.csv')
    tok = AutoTokenizer.from_pretrained(model_name)
    def prep_bin(df):
        texts = df['text'].astype(str).tolist(); labels = df['label_stage1'].values
        enc = tok(texts, truncation=True, padding=False, max_length=256); enc['labels'] = labels.tolist()
        return texts, labels, Dataset.from_dict(enc)
    def prep_4(df):
        texts = df['text'].astype(str).tolist(); labels = df['label_stage2'].map({'Liga Inggris':0,'Liga Indonesia':1,'Liga Spanyol':2,'Liga Italia':3}).values
        enc = tok(texts, truncation=True, padding=False, max_length=256); enc['labels'] = labels.tolist()
        return texts, labels, Dataset.from_dict(enc)
    t1_texts, y1_train, ds1_train = prep_bin(train1)
    v1_texts, y1_val, ds1_val = prep_bin(val1)
    s1_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    args1 = TrainingArguments(output_dir='experiments/llm_2stage/stage1', learning_rate=lr, per_device_train_batch_size=bs, per_device_eval_batch_size=bs, num_train_epochs=num_epochs, evaluation_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, report_to=[], lr_scheduler_type='linear', warmup_ratio=0.1)
    coll = DataCollatorWithPadding(tokenizer=tok)
    tr1 = Trainer(model=s1_model, args=args1, train_dataset=ds1_train, eval_dataset=ds1_val, tokenizer=tok, data_collator=coll)
    tr1.train()
    val1_pred = tr1.predict(ds1_val).predictions
    # probs untuk kelas sepakbola (index 1)
    val1_prob = torch.softmax(torch.tensor(val1_pred), dim=1)[:,1].numpy()
    th, _ = optimal_threshold(y1_val, val1_prob)
    print('Stage1 threshold:', th)
    # Stage2 model
    t2_texts, y2_train, ds2_train = prep_4(train2)
    v2_texts, y2_val, ds2_val = prep_4(val2)
    s2_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
    args2 = TrainingArguments(output_dir='experiments/llm_2stage/stage2', learning_rate=lr, per_device_train_batch_size=bs, per_device_eval_batch_size=bs, num_train_epochs=num_epochs, evaluation_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True, report_to=[], lr_scheduler_type='linear', warmup_ratio=0.1)
    tr2 = Trainer(model=s2_model, args=args2, train_dataset=ds2_train, eval_dataset=ds2_val, tokenizer=tok, data_collator=coll)
    tr2.train()
    
    # Pipeline prediksi untuk 5 kelas
    def predict_5(texts):
        # Create a dataset for prediction
        pred_df = pd.DataFrame({'text': texts})
        ds1 = Dataset.from_pandas(pred_df)
        
        # Tokenize the dataset
        def tokenize_function(examples):
            return tok(examples['text'], truncation=True, padding=False, max_length=256)
        
        ds1_tokenized = ds1.map(tokenize_function, batched=True)
        
        p1 = tr1.predict(ds1_tokenized).predictions
        prob1 = torch.softmax(torch.tensor(p1), dim=1)[:,1].numpy()
        is_fb = prob1 >= th
        preds = []
        
        fb_texts = [text for text, is_football in zip(texts, is_fb) if is_football]
        
        if fb_texts:
            pred_df_fb = pd.DataFrame({'text': fb_texts})
            ds2 = Dataset.from_pandas(pred_df_fb)
            ds2_tokenized = ds2.map(tokenize_function, batched=True)
            p2 = np.argmax(tr2.predict(ds2_tokenized).predictions, axis=1)
        else:
            p2 = np.array([])
            
        idx = 0
        for flag in is_fb:
            if flag: 
                preds.append(int(p2[idx]))
                idx += 1
            else: 
                preds.append(4)
        return np.array(preds)

    # Evaluasi
    train5 = pd.read_csv('dataset/train_5class.csv'); val5 = pd.read_csv('dataset/val_5class.csv'); test5 = pd.read_csv('dataset/test_5class.csv')
    def eval_df(df):
        texts = df['text'].astype(str).tolist(); y_true = df['label_5class'].map(LEAGUE_TO_IDX).values
        y_pred = predict_5(texts)
        rep = classification_report_np(y_true, y_pred, LABELS_5CLASS)
        cm = confusion_matrix_np(y_true, y_pred, len(LABELS_5CLASS))
        return rep, cm
    rep_train, cm_train = eval_df(train5)
    rep_val, cm_val = eval_df(val5)
    rep_test, cm_test = eval_df(test5)
    return {
        'train': rep_train, 'val': rep_val, 'test': rep_test,
        'cm_train': cm_train, 'cm_val': cm_val, 'cm_test': cm_test
    }

llm_2_results = run_llm_2stage()
llm_2_results['val']['accuracy'], llm_2_results['test']['accuracy']


In [ ]:
# Visualisasi perbandingan performa (contoh bar chart dari summary)
def plot_bar_comparison(df, title):
    metrics = ['val_accuracy', 'test_accuracy', 'val_f1_weighted', 'test_f1_weighted']
    plt.figure(figsize=(10,5))
    for i, m in enumerate(metrics):
        plt.subplot(2,2,i+1)
        sns.barplot(x='run', y=m, data=df)
        plt.xticks(rotation=45, ha='right'); plt.title(m)
    plt.suptitle(title); plt.tight_layout(); plt.show()

plot_bar_comparison(bilstm_1_summary, 'BiLSTM 1-Stage Metrics')
plot_bar_comparison(llm_1_summary, 'LLM 1-Stage Metrics')


## Penjelasan & Rekomendasi
- Hyperparameter: Pada BiLSTM, variasi `embed_dim` dan `lstm_units` mempengaruhi kapasitas representasi; unit lebih besar biasanya meningkatkan akurasi tetapi berisiko overfitting dan waktu latih lebih lama. Pada LLM, `learning_rate` dan `batch_size` mempengaruhi stabilitas fine-tuning; LR terlalu besar menurunkan umumisasi, sedangkan batch besar memperhalus update namun memerlukan memori lebih. Scheduler linear dengan warmup membantu stabilisasi awal pelatihan.
- Trade-off: BiLSTM lebih ringan, latih cepat, cocok untuk resource terbatas; LLM lebih akurat pada teks berbahasa campuran/nuansa semantik tetapi lebih berat waktu latih dan inferensi.
- Interpretasi metrics: F1 tertimbang mencerminkan performa pada distribusi kelas tidak seimbang. Confusion matrix mengungkap kelas yang sering tertukar (misal Liga Inggris vs Liga Spanyol).
- Rekomendasi: Jika tujuan utama akurasi, gunakan LLM 2-Stage (mengurangi kebingungan non-sepakbola) dengan threshold hasil validasi. Jika keterbatasan komputasi, gunakan BiLSTM 2-Stage. Untuk inference cepat di produksi, pertimbangkan 2-Stage karena memfilter dokumen non-sepakbola lebih awal.